# 07 — Sentinel-2 Verisi (alt bölgeler, 10 m)

**Amaç:** GeoClimate dokümanındaki **Sentinel-2 + U-Net** adımı için veri hazırlamak. Landsat (30 m) yerine Sentinel-2 (10 m) kullanılır;
derin öğrenme modeli (U-Net, notebook 08) pikselleri tek tek değil, **komşularıyla birlikte** görüntü parçası olarak sınıflandırır.

**Neden alt bölgeler?** Bütün ili 10 m'de işlemek Earth Engine kotasını ve Colab süresini aşar. Bu yüzden her ilde ilin araştırma konusunu temsil eden
**2 alt bölge** (20 × 20 km) seçildi (`config.S2_ALT_BOLGELER`): İstanbul'da kent çeperi, Artvin'de Yusufeli barajı ve ormanlık havza, Konya'da tarım ovaları,
Van'da göl kıyısı, Şanlıurfa'da GAP ovaları… Toplam 14 kutu.

**Akış:**
1. Kutuları oluştur ve haritada göster.
2. Her kutu ve yıl için kaç Sentinel-2 görüntüsü var? → **İlk yıl** = 14 kutunun hepsinde yaz verisi olan en erken yıl (2017, olmazsa 2018/2019).
3. Earth Engine'den Drive'a GeoTIFF: her kutu için **ilk yıl, 2021 (etiket yılı) ve 2025** yaz kompozitleri + **ESA WorldCover 2021** etiketi (10 m).
4. Kontrol: boyut, verisiz piksel oranı, sınıf dağılımı, örnek görüntüler.

**Süre ve kota:** 56 küçük görev; tahminen 15–40 dakika ve aylık kotanın küçük bir kısmı. Yarıda kalırsa hücreyi tekrar çalıştırın: biten görevler atlanır.
**GPU gerekmez** (CPU çalışma zamanı yeterli). Sonraki adım: `08_unet_siniflandirma` (GPU).

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Repo herkese açık olduğu için **token gerekmez**. Colab'ın **Gizli anahtarlar (Secrets)** bölümünde `GITHUB_TOKEN` tanımlıysa yine kullanılır (repo gizli yapılırsa gerekir); token ekrana yazdırılmaz.

In [ ]:
!pip install -q earthengine-api geemap rasterio

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
GITHUB_BRANCH = "main"   # kendi dalınızdaki kodu çalıştırmak için dal adını yazın (ör. "amine/geoclimate-uyum")
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        mesaj = r.stderr or r.stdout
        raise RuntimeError("git hatası: " + (mesaj.replace(token, "***") if token else mesaj))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    # Repo herkese açık: token gerekmez. Colab Secrets'ta GITHUB_TOKEN varsa kullanılır (repo gizli yapılırsa gerekir).
    try:
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        token = ""
    adres = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
    if not os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["clone", "-q", adres, CODE_DIR], token)
    # Seçilen dalın en güncel hâline geç (Colab kopyasındaki yerel değişiklikler atılır)
    _git(["-C", CODE_DIR, "fetch", "-q", "origin", GITHUB_BRANCH], token)
    _git(["-C", CODE_DIR, "checkout", "-q", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| dal:", GITHUB_BRANCH if IN_COLAB else "yerel", "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import ee
from src import gee_utils as gu
importlib.reload(gu)
gu.init_ee()

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
T, F = config.TABLES_DIR, config.FIGURES_DIR
RAW = f"{config.DATA_RAW}/s2"
os.makedirs(RAW, exist_ok=True)
SINIF_ADI = [config.CLASSES[k] for k in sorted(config.CLASSES)]
SINIF_RENK = [config.CLASS_COLORS[k] for k in sorted(config.CLASSES)]

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{F}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

## 1. Alt bölgeler (kutular)
Her kutunun köşeleri EPSG:3035 ızgarasına (10 m) oturtulur; böylece farklı yılların görüntüleri **piksel piksel aynı** yere düşer ve değişim doğrudan karşılaştırılabilir.
Haritada kutuların ilin araştırma konusunu gerçekten temsil ettiğini kontrol edin. Gerekirse `config.S2_ALT_BOLGELER`'de merkezleri değiştirin.

In [ ]:
KUTULAR = {}
satirlar = []
for il, liste in config.S2_ALT_BOLGELER.items():
    for ad, lat, lon in liste:
        kutu = f"{il}_{ad}"
        KUTULAR[kutu] = gu.s2_kutu(lat, lon)
        satirlar.append({"kutu": kutu, "il": il, "alt_bolge": ad, "merkez_enlem": lat, "merkez_boylam": lon,
                         "boyut_km": config.S2_KUTU_KM})
kutu_tab = pd.DataFrame(satirlar)
kutu_tab.to_csv(f"{T}/s2_alt_bolgeler.csv", index=False, encoding="utf-8-sig")

display(kutu_tab)
# Harita yalnızca görsel kontrol içindir; çizilemezse veri adımları etkilenmez
try:
    m = gu.make_map()
    m.addLayer(gu.get_city_boundaries().style(color="ffffff", fillColor="00000000", width=1), {}, "İller")
    for kutu, g in KUTULAR.items():
        m.addLayer(ee.FeatureCollection([ee.Feature(g)]).style(color="ff0000", fillColor="ff000033", width=2), {}, kutu)
    display(m)
except Exception as e:
    print(f"Harita çizilemedi ({type(e).__name__}: {e}). Veri adımlarına devam edebilirsiniz.")

## 2. Hangi yıllarda veri var?
Sentinel-2'nin atmosfer düzeltmeli (Level-2A) arşivi Türkiye için 2017'de eksik olabilir. Her kutu ve aday yıl için bulutluluğu %60'ın altındaki yaz görüntüleri sayılır.
**İlk yıl**, 14 kutunun hepsinde en az 3 görüntü olan en erken aday yıldır; seçim `data/processed/s2_yillar.json` dosyasına yazılır ve 08 bu dosyayı okur.

In [ ]:
yillar_aday = config.S2_ILK_YIL_ADAYLARI + [config.S2_ETIKET_YILI, config.S2_SON_YIL]
sayim = {kutu: ee.Dictionary({str(y): gu.s2_goruntu_sayisi(g, y) for y in yillar_aday}) for kutu, g in KUTULAR.items()}
sayim = pd.DataFrame(ee.Dictionary(sayim).getInfo()).T[[str(y) for y in yillar_aday]].astype(int)
sayim.to_csv(f"{T}/s2_goruntu_sayilari.csv", encoding="utf-8-sig")
display(sayim)

ILK_YIL = next((y for y in config.S2_ILK_YIL_ADAYLARI if (sayim[str(y)] >= 3).all()), None)
if ILK_YIL is None:
    raise RuntimeError("Aday yılların hiçbirinde 14 kutunun hepsinde yeterli görüntü yok. Tabloyu bize gönderin.")
for y in (config.S2_ETIKET_YILI, config.S2_SON_YIL):
    if (sayim[str(y)] < 3).any():
        raise RuntimeError(f"{y} için bazı kutularda 3'ten az görüntü var: {sayim.index[sayim[str(y)] < 3].tolist()}")
YILLAR = [ILK_YIL, config.S2_ETIKET_YILI, config.S2_SON_YIL]
json.dump({"ilk_yil": ILK_YIL, "etiket_yili": config.S2_ETIKET_YILI, "son_yil": config.S2_SON_YIL, "kutular": list(KUTULAR)},
          open(f"{config.DATA_PROCESSED}/s2_yillar.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("Kullanılacak yıllar:", YILLAR, "| değişim analizi:", ILK_YIL, "→", config.S2_SON_YIL)

## 3. Drive'a GeoTIFF export
Her kutu için 3 yaz kompoziti (6 bant, yansıma × 10000, int16) ve 1 etiket görüntüsü (WorldCover 2021 → 6 sınıf, eşlenmeyen = 255).
Dosyalar önce `Drive'ım/yapay-zeka-proje-gee/` klasörüne düşer, sonra otomatik olarak `data/raw/s2/` altına kopyalanır.

**Yarıda kalırsa:** Hücreyi tekrar çalıştırın. Biten görevler atlanır, çalışanlar beklenir, yalnızca başlamamış ya da başarısız olanlar yeniden başlatılır.

In [ ]:
exports = {}
for kutu, g in KUTULAR.items():
    for y in YILLAR:
        exports[f"s2_{kutu}_{y}"] = (gu.s2_kompozit(g, y), g)
    exports[f"s2_{kutu}_etiket"] = (gu.s2_etiket().clip(g), g)
print(len(exports), "görev")
kullanim = gu.run_image_exports(exports, RAW)
if kullanim is not None:
    display(kullanim)

## 4. Kontrol
Her dosya 2000 × 2000 piksel olmalı. **Verisiz piksel** (bulut nedeniyle hiç temiz gözlem yok) oranı yüksek kutuları not edin: bu pikseller U-Net eğitiminde ve
değerlendirmede yok sayılır. Sınıf dağılımı, kutunun ilin konusunu temsil edip etmediğini gösterir (ör. Konya'da tarım, Van'da su payı yüksek olmalı).

In [ ]:
satirlar = []
for kutu in KUTULAR:
    with rasterio.open(f"{RAW}/s2_{kutu}_etiket.tif") as r:
        etiket = r.read(1)
    say = np.bincount(etiket.ravel(), minlength=256)
    satir = {"kutu": kutu, "boyut": f"{etiket.shape[0]}x{etiket.shape[1]}",
             "etiketsiz_yuzde": round(100 * say[config.S2_ETIKET_YOK] / etiket.size, 2)}
    for k, ad in zip(sorted(config.CLASSES), SINIF_ADI):
        satir[f"{ad}_yuzde"] = round(100 * say[k] / etiket.size, 1)
    for y in YILLAR:
        with rasterio.open(f"{RAW}/s2_{kutu}_{y}.tif") as r:
            x = r.read()
            assert x.shape == (len(config.S2_BANDS),) + etiket.shape, (kutu, y, x.shape)
        satir[f"verisiz_{y}_yuzde"] = round(100 * (x == config.S2_NODATA).all(axis=0).mean(), 2)
    satirlar.append(satir)
ozet = pd.DataFrame(satirlar).set_index("kutu")
ozet.to_csv(f"{T}/s2_veri_ozeti.csv", encoding="utf-8-sig")
display(ozet)
if (ozet[[f"verisiz_{y}_yuzde" for y in YILLAR]] > 20).any().any():
    print("UYARI: bazı kutularda verisiz piksel %20'nin üstünde; bu kutuların sonuçlarını temkinli yorumlayın.")

In [ ]:
from matplotlib.colors import ListedColormap

def rgb(kutu, y):
    with rasterio.open(f"{RAW}/s2_{kutu}_{y}.tif") as r:
        x = r.read([3, 2, 1]).astype(float)                        # B4, B3, B2 = kırmızı, yeşil, mavi
    return np.clip(np.moveaxis(x, 0, -1) / 3000, 0, 1)

renk = ListedColormap(SINIF_RENK)
fig, axes = plt.subplots(len(KUTULAR), 4, figsize=(13, 3.2 * len(KUTULAR)))
for satir, kutu in zip(axes, KUTULAR):
    with rasterio.open(f"{RAW}/s2_{kutu}_etiket.tif") as r:
        e = r.read(1).astype(float)
    e[e == config.S2_ETIKET_YOK] = np.nan
    for ax, (baslik, img) in zip(satir, [(f"{ILK_YIL}", rgb(kutu, ILK_YIL)), (f"{config.S2_ETIKET_YILI}", rgb(kutu, config.S2_ETIKET_YILI)),
                                         (f"{config.S2_SON_YIL}", rgb(kutu, config.S2_SON_YIL))]):
        ax.imshow(img); ax.set_title(f"{kutu} · {baslik}", fontsize=8); ax.axis("off")
    satir[3].imshow(e, cmap=renk, vmin=-0.5, vmax=5.5, interpolation="nearest")
    satir[3].set_title("WorldCover 2021 etiketi", fontsize=8); satir[3].axis("off")
savefig("s2_kutular")

## Faz bitti mi?
- [ ] `results/tables/s2_alt_bolgeler.csv`, `s2_goruntu_sayilari.csv`, `s2_veri_ozeti.csv`
- [ ] `data/raw/s2/` altında 14 × 4 = 56 GeoTIFF; `data/processed/s2_yillar.json`
- [ ] `results/figures/s2_kutular.png`: görüntüler bulutsuz ve kutular doğru yerde mi?

## Bu fazda ne yaptık, neden yaptık
- Landsat yerine **Sentinel-2 (10 m)** kullandık: üç kat daha keskin; tarla sınırları, kent dokusu ve kıyı çizgisi daha net.
- Bütün iller yerine ilin konusunu temsil eden **14 alt bölge** seçtik: 10 m'de bütün iller kotayı ve süreyi aşardı.
- Bulutları iki yolla maskeledik (s2cloudless olasılığı ve sahne sınıflandırma katmanı SCL) ve yaz medyanını aldık; aynı mevsim karşılaştırılmazsa
  mevsim farkı "arazi değişimi" gibi görünür.
- Bütün yılların görüntülerini **aynı 10 m ızgarada** ürettik; değişim piksel piksel karşılaştırılabilir.
- Etiket olarak yine **ESA WorldCover 2021** ve aynı 6 sınıfı kullandık; Landsat modelleriyle (03) karşılaştırma adil olsun diye.